In [30]:
import os
from torchvision import datasets , transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Load Data

In [4]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness = 0.2 , contrast = 0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean =[0.485 , 0.456 , 0.406] , std =[0.229 , 0.224 , 0.225])  # from imagenet
])

In [5]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(dataset_path , transform = image_transforms)
len(dataset)

2300

In [6]:
dataset.classes

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [7]:
num_classes = len(dataset.classes)
num_classes

6

In [8]:
train_size = int(0.75 * len(dataset))
val_size = len(dataset) - train_size
train_size , val_size

(1725, 575)

In [9]:
from torch.utils.data import random_split

train_dataset , val_dataset = random_split(dataset , [train_size , val_size])

In [10]:
train_loader = DataLoader(train_dataset , batch_size=32 , shuffle = True)
val_loader = DataLoader(val_dataset , batch_size=32 , shuffle=True)

In [11]:
def train_model(model , criterion , optimizer , epochs = 5):
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num , (images , labels) in enumerate(train_loader):
            images , labels = images.to(device) , labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            #forward pass
            outputs = model(images)
            loss = criterion(outputs , labels)
            
            #backword pass and optimizer
            loss.backward()
            optimizer.step()

            if (batch_num+1) % 10 == 0:
                print(f"Batch : {batch_num+1} , Epoch: {epoch+1} , Loss: {loss.item(): 0.2f}")

            running_loss += loss.item() * images.size(0)
            
        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] , Avg loss: {epoch_loss: .4f}")

        # validation
        model.eval()
        correct=0
        total=0
        all_labels = []
        all_predictions =[]

        with torch.no_grad():
            for images , labels in val_loader:
                images , labels = images.to(device) , labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data,1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())
            print(f"*** Validation Accuracy : {100 * correct / total:.2f}% ***")
    end = time.time()
    print(f"Execution time : {end-start} seconds")
    return all_labels , all_predictions      

In [14]:
class CarClassifierEfficientNet(nn.Module):
    def __init__(self , num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [13]:
model = CarClassifierEfficientNet(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.001)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.56
Batch : 20 , Epoch: 1 , Loss:  1.63
Batch : 30 , Epoch: 1 , Loss:  1.34
Batch : 40 , Epoch: 1 , Loss:  1.19
Batch : 50 , Epoch: 1 , Loss:  1.44
Epoch [1/10] , Avg loss:  1.4961
*** Validation Accuracy : 60.87% ***
Batch : 10 , Epoch: 2 , Loss:  1.14
Batch : 20 , Epoch: 2 , Loss:  1.05
Batch : 30 , Epoch: 2 , Loss:  0.88
Batch : 40 , Epoch: 2 , Loss:  1.05
Batch : 50 , Epoch: 2 , Loss:  1.01
Epoch [2/10] , Avg loss:  1.1487
*** Validation Accuracy : 64.17% ***
Batch : 10 , Epoch: 3 , Loss:  1.12
Batch : 20 , Epoch: 3 , Loss:  0.93
Batch : 30 , Epoch: 3 , Loss:  0.87
Batch : 40 , Epoch: 3 , Loss:  1.10
Batch : 50 , Epoch: 3 , Loss:  1.07
Epoch [3/10] , Avg loss:  1.0209
*** Validation Accuracy : 66.61% ***
Batch : 10 , Epoch: 4 , Loss:  0.95
Batch : 20 , Epoch: 4 , Loss:  0.96
Batch : 30 , Epoch: 4 , Loss:  1.05
Batch : 40 , Epoch: 4 , Loss:  0.98
Batch : 50 , Epoch: 4 , Loss:  1.01
Epoch [4/10] , Avg loss:  0.9747
*** Validation Accuracy : 67.30% ***


### Hyperparameter Tuning

In [34]:
class CarClassifierEfficientNet(nn.Module):
    def __init__(self , num_classes , dropout_rate):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [35]:
# define objective function for optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr' , 1e-5 , 1e-2 , log=True)
    dropout_rate = trial.suggest_float('dropout_rate' , 0.2 , 0.7)

    # load the model
    model = CarClassifierEfficientNet(num_classes = num_classes , dropout_rate=dropout_rate).to(device)

    #difine the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad , model.parameters()), lr = lr)

    # Training loop
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num ,(images , labels) in enumerate(train_loader):
            images, labels = images.to(device) , labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs , labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images , labels in val_loader:
                images , labels = images.to(device) , labels.to(device)
                outputs = model(images)
                _,predicted = torch.max(outputs.data , 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct/total
        
        # report imidiate results to optuna
        trial.report(accuracy , epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time : {int((end-start)/60)} minutes and {(end-start)%60} seconds ")

    return accuracy
        
                

In [36]:
#create the study and optimize
study = optuna.create_study(direction ='maximize') 
study.optimize(objective , n_trials=20)

[I 2025-12-27 22:32:25,174] A new study created in memory with name: no-name-21de9ea7-202b-4bf4-8c52-1e3a4ffe5e3a
[I 2025-12-27 22:41:59,459] Trial 0 finished with value: 55.130434782608695 and parameters: {'lr': 0.00021072669954549345, 'dropout_rate': 0.6023306375638475}. Best is trial 0 with value: 55.130434782608695.


Execution time : 9 minutes and 34.04636883735657 seconds 


[I 2025-12-27 22:52:12,586] Trial 1 finished with value: 66.78260869565217 and parameters: {'lr': 0.003861079521799264, 'dropout_rate': 0.5611350705292062}. Best is trial 1 with value: 66.78260869565217.


Execution time : 10 minutes and 12.918305158615112 seconds 


[I 2025-12-27 23:02:19,921] Trial 2 finished with value: 64.8695652173913 and parameters: {'lr': 0.0065354821183583305, 'dropout_rate': 0.5033732578366081}. Best is trial 1 with value: 66.78260869565217.


...

[I 2025-12-28 00:01:31,565] Trial 12 pruned. 
[I 2025-12-28 00:04:43,793] Trial 13 pruned. 
[I 2025-12-28 00:14:57,940] Trial 14 finished with value: 68.17391304347827 and parameters: {'lr': 0.0026619235118782994, 'dropout_rate': 0.39391807120356204}. Best is trial 14 with value: 68.17391304347827.


Execution time : 10 minutes and 13.947465419769287 seconds 


[I 2025-12-28 00:24:40,018] Trial 15 pruned. 
[I 2025-12-28 00:27:49,709] Trial 16 pruned. 
[I 2025-12-28 00:37:39,522] Trial 17 finished with value: 66.78260869565217 and parameters: {'lr': 0.003472754145720044, 'dropout_rate': 0.43628482749120945}. Best is trial 14 with value: 68.17391304347827.


Execution time : 9 minutes and 49.64425230026245 seconds 


[I 2025-12-28 00:40:52,949] Trial 18 pruned. 
[I 2025-12-28 00:44:08,011] Trial 19 pruned. 


In [37]:
best_params = study.best_params
print(best_params)

{'lr': 0.0026619235118782994, 'dropout_rate': 0.39391807120356204}


In [38]:
model = CarClassifierEfficientNet(num_classes=num_classes , dropout_rate = 0.4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.0026)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.37
Batch : 20 , Epoch: 1 , Loss:  1.41
Batch : 30 , Epoch: 1 , Loss:  1.20
Batch : 40 , Epoch: 1 , Loss:  1.12
Batch : 50 , Epoch: 1 , Loss:  1.00
Epoch [1/10] , Avg loss:  1.3020
*** Validation Accuracy : 63.83% ***
Batch : 10 , Epoch: 2 , Loss:  0.81
Batch : 20 , Epoch: 2 , Loss:  0.77
Batch : 30 , Epoch: 2 , Loss:  1.17
Batch : 40 , Epoch: 2 , Loss:  0.99
Batch : 50 , Epoch: 2 , Loss:  0.79
Epoch [2/10] , Avg loss:  0.9368
*** Validation Accuracy : 66.09% ***
Batch : 10 , Epoch: 3 , Loss:  0.92
Batch : 20 , Epoch: 3 , Loss:  0.89
Batch : 30 , Epoch: 3 , Loss:  1.10
Batch : 40 , Epoch: 3 , Loss:  0.83
Batch : 50 , Epoch: 3 , Loss:  0.77
Epoch [3/10] , Avg loss:  0.8572
*** Validation Accuracy : 68.87% ***
Batch : 10 , Epoch: 4 , Loss:  0.77
Batch : 20 , Epoch: 4 , Loss:  0.70
Batch : 30 , Epoch: 4 , Loss:  0.88
Batch : 40 , Epoch: 4 , Loss:  0.78
Batch : 50 , Epoch: 4 , Loss:  0.71
Epoch [4/10] , Avg loss:  0.8154
*** Validation Accuracy : 69.04% ***


In [17]:
class CarClassifierEfficientNetf2(nn.Module):
    def __init__(self , num_classes):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze last feature block
        for param in self.model.features[-1].parameters():
            param.requires_grad = True

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x       

In [18]:
model = CarClassifierEfficientNetf2(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.001)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.55
Batch : 20 , Epoch: 1 , Loss:  1.29
Batch : 30 , Epoch: 1 , Loss:  1.23
Batch : 40 , Epoch: 1 , Loss:  1.03
Batch : 50 , Epoch: 1 , Loss:  0.97
Epoch [1/10] , Avg loss:  1.2149
*** Validation Accuracy : 66.43% ***
Batch : 10 , Epoch: 2 , Loss:  1.10
Batch : 20 , Epoch: 2 , Loss:  0.63
Batch : 30 , Epoch: 2 , Loss:  0.69
Batch : 40 , Epoch: 2 , Loss:  0.72
Batch : 50 , Epoch: 2 , Loss:  0.65
Epoch [2/10] , Avg loss:  0.8016
*** Validation Accuracy : 69.04% ***
Batch : 10 , Epoch: 3 , Loss:  0.59
Batch : 20 , Epoch: 3 , Loss:  0.53
Batch : 30 , Epoch: 3 , Loss:  0.49
Batch : 40 , Epoch: 3 , Loss:  0.75
Batch : 50 , Epoch: 3 , Loss:  0.59
Epoch [3/10] , Avg loss:  0.6616
*** Validation Accuracy : 70.09% ***
Batch : 10 , Epoch: 4 , Loss:  0.76
Batch : 20 , Epoch: 4 , Loss:  0.38
Batch : 30 , Epoch: 4 , Loss:  0.67
Batch : 40 , Epoch: 4 , Loss:  0.45
Batch : 50 , Epoch: 4 , Loss:  0.46
Epoch [4/10] , Avg loss:  0.6023
*** Validation Accuracy : 69.39% ***


### Hyperparameter tuning

In [25]:
class CarClassifierEfficientNetf2(nn.Module):
    def __init__(self , num_classes , dropout_rate):
        super().__init__()
        self.model = models.efficientnet_b0(weights='DEFAULT')

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze last feature block
        for param in self.model.features[-1].parameters():
            param.requires_grad = True

        in_features = self.model.classifier[1].in_features

        self.model.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features , num_classes)
        )

    def forward(self , x):
        x = self.model(x)
        return x 

        

In [27]:
# define objective function for optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr' , 1e-5 , 1e-2 , log=True)
    dropout_rate = trial.suggest_float('dropout_rate' , 0.2 , 0.7)

    # load the model
    model = CarClassifierEfficientNetf2(num_classes = num_classes , dropout_rate=dropout_rate).to(device)

    #difine the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad , model.parameters()), lr = lr)

    # Training loop
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num ,(images , labels) in enumerate(train_loader):
            images, labels = images.to(device) , labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs , labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images , labels in val_loader:
                images , labels = images.to(device) , labels.to(device)
                outputs = model(images)
                _,predicted = torch.max(outputs.data , 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct/total
        
        # report imidiate results to optuna
        trial.report(accuracy , epoch)

        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time : {int((end-start)/60)} minutes and {(end-start)%60} seconds ")

    return accuracy
        
                

In [31]:
#create the study and optimize
study = optuna.create_study(direction ='maximize') 
study.optimize(objective , n_trials=20)

[I 2025-12-26 22:01:10,029] A new study created in memory with name: no-name-1284eba8-0d94-4d37-b1cb-89c85f66e06f
[I 2025-12-26 22:14:40,724] Trial 0 finished with value: 69.3913043478261 and parameters: {'lr': 0.009921848054274854, 'dropout_rate': 0.6916346129028479}. Best is trial 0 with value: 69.3913043478261.


Execution time : 13 minutes and 30.207653522491455 seconds 


[I 2025-12-26 22:24:10,708] Trial 1 finished with value: 66.08695652173913 and parameters: {'lr': 0.00031897681338046953, 'dropout_rate': 0.5474954101034807}. Best is trial 0 with value: 69.3913043478261.


Execution time : 9 minutes and 29.71520495414734 seconds 


[I 2025-12-26 22:33:29,105] Trial 2 finished with value: 46.08695652173913 and parameters: {'lr': 3.605504149253511e-05, 'dropout_rate': 0.22709386685326055}. Best is trial 0 with value: 69.3913043478261.


...

[I 2025-12-26 23:57:07,226] Trial 14 finished with value: 72.0 and parameters: {'lr': 0.0017051661176909903, 'dropout_rate': 0.2996491457334073}. Best is trial 14 with value: 72.0.


Execution time : 9 minutes and 38.292235136032104 seconds 


[I 2025-12-27 00:00:26,764] Trial 15 pruned. 
[I 2025-12-27 00:03:43,962] Trial 16 pruned. 
[I 2025-12-27 00:12:56,768] Trial 17 finished with value: 69.73913043478261 and parameters: {'lr': 0.0036669387418895565, 'dropout_rate': 0.277789700199904}. Best is trial 14 with value: 72.0.


Execution time : 9 minutes and 11.59702205657959 seconds 


[I 2025-12-27 00:16:05,901] Trial 18 pruned. 
[I 2025-12-27 00:19:08,257] Trial 19 pruned. 


In [32]:
best_params = study.best_params
print(best_params)

{'lr': 0.0017051661176909903, 'dropout_rate': 0.2996491457334073}


In [33]:
model = CarClassifierEfficientNetf2(num_classes=num_classes , dropout_rate = 0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p:p.requires_grad , model.parameters()) , lr =0.0017)
 
labels , predictions = train_model(model , criterion , optimizer , epochs =10)

Batch : 10 , Epoch: 1 , Loss:  1.38
Batch : 20 , Epoch: 1 , Loss:  0.92
Batch : 30 , Epoch: 1 , Loss:  0.73
Batch : 40 , Epoch: 1 , Loss:  0.83
Batch : 50 , Epoch: 1 , Loss:  0.74
Epoch [1/10] , Avg loss:  1.0686
*** Validation Accuracy : 66.61% ***
Batch : 10 , Epoch: 2 , Loss:  0.77
Batch : 20 , Epoch: 2 , Loss:  0.50
Batch : 30 , Epoch: 2 , Loss:  0.67
Batch : 40 , Epoch: 2 , Loss:  0.67
Batch : 50 , Epoch: 2 , Loss:  0.68
Epoch [2/10] , Avg loss:  0.6903
*** Validation Accuracy : 70.26% ***
Batch : 10 , Epoch: 3 , Loss:  0.68
Batch : 20 , Epoch: 3 , Loss:  0.46
Batch : 30 , Epoch: 3 , Loss:  0.54
Batch : 40 , Epoch: 3 , Loss:  0.33
Batch : 50 , Epoch: 3 , Loss:  0.75
Epoch [3/10] , Avg loss:  0.5775
*** Validation Accuracy : 72.87% ***
Batch : 10 , Epoch: 4 , Loss:  0.53
Batch : 20 , Epoch: 4 , Loss:  0.54
Batch : 30 , Epoch: 4 , Loss:  0.43
Batch : 40 , Epoch: 4 , Loss:  0.64
Batch : 50 , Epoch: 4 , Loss:  0.40
Epoch [4/10] , Avg loss:  0.4872
*** Validation Accuracy : 71.83% ***
